# SimpleCNN Standalone Colab Notebook

This notebook trains SimpleCNN from scratch on FER2013.
It is fully standalone and does not depend on the repo's Python modules.

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('Colab environment:', IN_COLAB)
print('Torch version:', torch.__version__)

## Set Up Colab Runtime and Dataset

Put FER2013 in `data/fer2013` at the notebook root.
If you use Google Drive, copy the folder there before running training cells.

In [ ]:
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data' / 'fer2013'
SAVE_DIR = ROOT / 'saved_models'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Expected FER2013 folder at: {DATA_DIR}')

print('Project root:', ROOT)
print('Data path:', DATA_DIR)
print('Save path:', SAVE_DIR)

## Load Dataset and Preview Samples

This notebook uses the existing FER2013 folder layout with `train` and `test` subfolders.
The validation split is carved out of the training folder inside the notebook.

In [ ]:
IMAGE_SIZE = 48
BATCH_SIZE = 64
NUM_CLASSES = 7
VAL_FRACTION = 0.1
SEED = 42

# Mild but stronger augmentation while still keeping a simple baseline setup.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12), ratio=(0.3, 3.3), value='random'),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

full_train_dataset = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)
test_dataset = datasets.ImageFolder(DATA_DIR / 'test', transform=test_transform)

class_names = full_train_dataset.classes
assert len(class_names) == NUM_CLASSES, class_names

val_size = int(len(full_train_dataset) * VAL_FRACTION)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

# Use deterministic class weighting to reduce class-imbalance bias.
train_labels = np.array([full_train_dataset.targets[idx] for idx in train_dataset.indices])
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights_np = class_counts.sum() / (NUM_CLASSES * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print('Classes:', class_names)
print('Train samples:', len(train_dataset))
print('Validation samples:', len(val_dataset))
print('Test samples:', len(test_dataset))
print('Class counts (train split):', class_counts.tolist())
print('Class weights:', [round(x, 4) for x in class_weights_np.tolist()])

sample_images, sample_labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, sample_images[:8], sample_labels[:8]):
    axis.imshow(image.squeeze(0), cmap='gray')
    axis.set_title(class_names[int(label)])
    axis.axis('off')
plt.tight_layout()

## Build the Baseline Simple CNN

This is a baseline CNN trained from scratch. No pretrained backbone is used.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout_fc = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.dropout(self.pool(torch.relu(self.bn1(self.conv1(x)))))
        x = self.dropout(self.pool(torch.relu(self.bn2(self.conv2(x)))))
        x = self.dropout(self.pool(torch.relu(self.bn3(self.conv3(x)))))
        x = self.dropout(self.pool(torch.relu(self.bn4(self.conv4(x)))))
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(self.dropout_fc(torch.relu(x)))
        return x


def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)

model = SimpleCNN(num_classes=NUM_CLASSES).to(DEVICE)
print('Trainable parameters:', f'{count_parameters(model):,}')

# Class-balanced and slightly smoothed CE can improve generalization without changing architecture.
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
USE_AMP = DEVICE.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

## Train the Baseline Model

This is the main run. It trains SimpleCNN from scratch, uses GPU if available, and saves the best checkpoint.

In [ ]:
EPOCHS = 80
PATIENCE = 12
MIN_IMPROVEMENT = 0.02
BEST_MODEL_PATH = SAVE_DIR / 'SimpleCNN_best_standalone.pth'

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_val_acc = 0.0
best_state_dict = None
best_epoch = 0
epochs_without_improve = 0


def to_device(batch_tensor):
    return batch_tensor.to(DEVICE, non_blocking=USE_AMP)


start_time = time.time()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, targets in tqdm(train_loader, desc=f'Train {epoch + 1}/{EPOCHS}', leave=False):
        inputs = to_device(inputs)
        targets = to_device(targets)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        if USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        train_correct += (outputs.argmax(dim=1) == targets).sum().item()
        train_total += targets.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = to_device(inputs)
            targets = to_device(targets)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(dim=1) == targets).sum().item()
            val_total += targets.size(0)

    scheduler.step()

    train_acc = 100.0 * train_correct / train_total
    val_acc = 100.0 * val_correct / val_total
    avg_train_loss = train_loss / train_total
    avg_val_loss = val_loss / val_total
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f'Epoch {epoch + 1:02d}/{EPOCHS} | '
        f'LR: {current_lr:.6f} | '
        f'Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
        f'Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%'
    )

    if val_acc > best_val_acc + MIN_IMPROVEMENT:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        epochs_without_improve = 0
        best_state_dict = model.state_dict()
        torch.save(best_state_dict, BEST_MODEL_PATH)
        print(f'Best checkpoint saved to {BEST_MODEL_PATH}')
    else:
        epochs_without_improve += 1

    if epochs_without_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch + 1} (no val improvement for {PATIENCE} epochs).')
        break

elapsed = time.time() - start_time
print(f'Best validation accuracy: {best_val_acc:.2f}% at epoch {best_epoch}')
print(f'Total training time: {elapsed / 60:.1f} minutes')

## Evaluate Baseline Performance

This cell reloads the best checkpoint and reports the test metrics.

In [ ]:
best_model = SimpleCNN(num_classes=NUM_CLASSES).to(DEVICE)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
best_model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = to_device(inputs)
        targets = to_device(targets)
        outputs = best_model(inputs)
        predictions = outputs.argmax(dim=1)
        y_true.extend(targets.cpu().numpy().tolist())
        y_pred.extend(predictions.cpu().numpy().tolist())

accuracy = (np.array(y_true) == np.array(y_pred)).mean() * 100.0
print(f'Test accuracy: {accuracy:.2f}%')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
plt.xticks(rotation=45)
plt.tight_layout()

## Final Notes

SimpleCNN is still the baseline deliverable and is trained from scratch.

Upgrades added while keeping it simple:
- class-balanced loss (to reduce class imbalance bias)
- mild label smoothing
- slightly stronger but still standard augmentation
- gradient clipping for more stable updates
- longer schedule with early stopping to avoid overtraining
- cosine learning-rate decay with lower minimum LR